# 01 — DGCA Email Pipeline (Phase 1)

**Project:** DGCA Agentic AI Regulatory Intelligence & Monitoring System
**Phase:** 1 of 8 — Email Generation & Delivery Pipeline

## 1. Project Overview

This notebook builds and tests the email notification pipeline with **email
generation deliberately decoupled from email delivery**. Getting an Azure AD
App Registration approved can be slow (it needs tenant-admin consent) and is
**not a blocker** for building and validating the rest of this system.

**Pipeline flow implemented in this notebook:**

```
Structured DGCA Update (data model)
        |
        v
AI Summary Agent            (OpenAI if configured, else deterministic fallback)
        |
        v
HTML Email Renderer         (professional, aviation-industry appropriate --
        |                     identical output regardless of delivery provider)
        v
Email Validator              (confirms every required field actually made it
        |                     into the rendered HTML before anything is sent)
        v
EmailSender interface  <-----  EMAIL_PROVIDER selects the implementation:
        |                        "preview"          -- local .html file (default, always works)
        |                        "microsoft_graph"   -- Outlook via Graph (production, optional)
        |                        "smtp"              -- generic SMTP (dev-friendly alternative)
        |                        "api"               -- reserved extension point
        v
TEST_MODE = True (default) --> even if a real provider is selected, delivery
                                is skipped and the email is only previewed.
```

**What this notebook deliberately does NOT do (out of scope for Phase 1):**
DGCA scraping (that's Phase 2, already done separately), LangGraph/agent
orchestration, old-vs-new PDF comparison, RAG, vector databases, the
Streamlit dashboard, OCR.

## Why decouple generation from delivery?

Email *generation* (data -> summary -> HTML -> validation) has no external
dependency and can be fully built and tested today. Email *delivery* is
provider-specific and, for the production provider (Microsoft Graph),
requires an Azure AD App Registration with `Mail.Send` application permission
and tenant-admin consent -- approvals that can take time and are entirely
outside this notebook's control. Splitting the two means:

- The generation/validation logic is provider-agnostic and testable right now.
- Microsoft Graph stays the planned production provider, implemented and
  ready, but gated behind configuration that simply isn't present yet.
- A working local preview (and an optional SMTP dev path) means this phase is
  demonstrably complete without waiting on anyone else.


## 2. Configuration

All configuration is read from environment variables, loaded from `config/.env`
(never committed — see `pipeline/config.py` for the settings). No section of
this notebook requires `config/.env` to exist at all -- every value has a safe
default (`EMAIL_PROVIDER=preview`, `TEST_MODE=true`).

| Variable | Purpose | Required for... |
|---|---|---|
| `EMAIL_PROVIDER` | `preview` \| `microsoft_graph` \| `smtp` \| `api` | selects the delivery provider (default: `preview`) |
| `TEST_MODE` | `true` (default, safe) or `false` | master safety switch — see below |
| `AZURE_CLIENT_ID` / `AZURE_TENANT_ID` / `AZURE_CLIENT_SECRET` | Azure AD App Registration | `microsoft_graph` provider only |
| `OUTLOOK_SENDER_EMAIL` / `OUTLOOK_RECIPIENT_EMAIL` | From/To mailbox | `microsoft_graph` and `smtp` providers |
| `SMTP_HOST` / `SMTP_PORT` / `SMTP_USERNAME` / `SMTP_PASSWORD` / `SMTP_USE_TLS` | SMTP connection | `smtp` provider only |
| `OPENAI_API_KEY` / `LLM_MODEL` | AI summary generation | optional — falls back to a template summary |

**Safety rule enforced by this notebook:** while `TEST_MODE=true` (the
default), even if `EMAIL_PROVIDER` is set to a real delivery provider, the
pipeline automatically falls back to the local preview instead of attempting
delivery. Secrets are never printed — only whether each variable is
`SET`/`MISSING` is logged.


## 3. Imports

In [ ]:
import json
import logging
import re
import smtplib
import time
import uuid
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from datetime import datetime, timezone
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from pathlib import Path
from typing import Callable, List, Optional, TypeVar

import requests
from dotenv import load_dotenv
from jinja2 import Template
from pydantic import BaseModel, Field

# msal (Microsoft Graph auth) and openai (AI summaries) are optional-in-spirit:
# both are installed (see requirements.txt), but neither is required to be
# *configured* for this notebook to run end-to-end.
import msal

try:
    from openai import OpenAI
    _OPENAI_SDK_AVAILABLE = True
except ImportError:  # pragma: no cover
    _OPENAI_SDK_AVAILABLE = False

print("Imports OK")


## 4. Environment Variables

`PROJECT_ROOT` is resolved relative to this notebook's location (`notebooks/..`),
so the notebook works regardless of the current working directory Jupyter was
launched from. `config/.env` is loaded explicitly from that root — if it
doesn't exist, every setting simply falls back to its safe default.


In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ENV_PATH = PROJECT_ROOT / "config" / ".env"
LOG_DIR = PROJECT_ROOT / "logs"
PREVIEW_DIR = PROJECT_ROOT / "data" / "processed" / "email_previews"

LOG_DIR.mkdir(parents=True, exist_ok=True)
PREVIEW_DIR.mkdir(parents=True, exist_ok=True)

env_loaded = load_dotenv(dotenv_path=ENV_PATH)
print(f"PROJECT_ROOT   : {PROJECT_ROOT}")
print(f"config/.env    : {'found and loaded' if env_loaded else 'NOT FOUND (using safe defaults only)'}")


In [ ]:
import os

_PLACEHOLDER_PATTERNS = ("your-", "00000000-0000-0000-0000-000000000000", "sk-your-")


def _is_placeholder(value: Optional[str]) -> bool:
    if not value:
        return True
    return any(value.startswith(p) for p in _PLACEHOLDER_PATTERNS)


@dataclass
class PipelineConfig:
    email_provider: str = "preview"
    test_mode: bool = True

    azure_client_id: Optional[str] = None
    azure_tenant_id: Optional[str] = None
    azure_client_secret: Optional[str] = None

    outlook_sender_email: Optional[str] = None
    outlook_recipient_email: Optional[str] = None

    smtp_host: Optional[str] = None
    smtp_port: int = 587
    smtp_username: Optional[str] = None
    smtp_password: Optional[str] = None
    smtp_use_tls: bool = True

    openai_api_key: Optional[str] = None
    llm_model: str = "gpt-4o-mini"

    def graph_configured(self) -> bool:
        return not any(
            _is_placeholder(v)
            for v in (
                self.azure_client_id,
                self.azure_tenant_id,
                self.azure_client_secret,
                self.outlook_sender_email,
                self.outlook_recipient_email,
            )
        )

    def smtp_configured(self) -> bool:
        return bool(self.smtp_host) and not _is_placeholder(self.outlook_sender_email)

    def openai_configured(self) -> bool:
        return not _is_placeholder(self.openai_api_key)


def load_config() -> PipelineConfig:
    return PipelineConfig(
        email_provider=os.getenv("EMAIL_PROVIDER", "preview").strip().lower(),
        test_mode=os.getenv("TEST_MODE", "true").strip().lower() != "false",
        azure_client_id=os.getenv("AZURE_CLIENT_ID"),
        azure_tenant_id=os.getenv("AZURE_TENANT_ID"),
        azure_client_secret=os.getenv("AZURE_CLIENT_SECRET"),
        outlook_sender_email=os.getenv("OUTLOOK_SENDER_EMAIL"),
        outlook_recipient_email=os.getenv("OUTLOOK_RECIPIENT_EMAIL"),
        smtp_host=os.getenv("SMTP_HOST"),
        smtp_port=int(os.getenv("SMTP_PORT", "587") or "587"),
        smtp_username=os.getenv("SMTP_USERNAME"),
        smtp_password=os.getenv("SMTP_PASSWORD"),
        smtp_use_tls=os.getenv("SMTP_USE_TLS", "true").strip().lower() != "false",
        openai_api_key=os.getenv("OPENAI_API_KEY"),
        llm_model=os.getenv("LLM_MODEL", "gpt-4o-mini"),
    )


config = load_config()

# Never print secret values -- only whether each is configured.
_status = {
    "AZURE_CLIENT_ID": not _is_placeholder(config.azure_client_id),
    "AZURE_TENANT_ID": not _is_placeholder(config.azure_tenant_id),
    "AZURE_CLIENT_SECRET": not _is_placeholder(config.azure_client_secret),
    "OUTLOOK_SENDER_EMAIL": not _is_placeholder(config.outlook_sender_email),
    "OUTLOOK_RECIPIENT_EMAIL": not _is_placeholder(config.outlook_recipient_email),
    "SMTP_USERNAME": bool(config.smtp_username),
    "SMTP_PASSWORD": bool(config.smtp_password),
    "OPENAI_API_KEY": not _is_placeholder(config.openai_api_key),
}
print("Configuration status (secret values never shown):")
print(f"  {'EMAIL_PROVIDER':<24} {config.email_provider}")
print(f"  {'TEST_MODE':<24} {config.test_mode}")
print(f"  {'SMTP_HOST':<24} {config.smtp_host or '(not set)'}")
print(f"  {'SMTP_PORT':<24} {config.smtp_port}")
print(f"  {'LLM_MODEL':<24} {config.llm_model}")
for key, is_set in _status.items():
    print(f"  {key:<24} {'SET' if is_set else 'MISSING (placeholder)'}")
print()
print(f"Microsoft Graph fully configured : {config.graph_configured()}")
print(f"SMTP fully configured             : {config.smtp_configured()}")
print(f"OpenAI fully configured           : {config.openai_configured()}")


## 5. Data Models

`DGCAUpdate` is the structured contract between the (future) scraper/AI-analysis
stages and this email pipeline — unchanged from the original design.


In [ ]:
class DGCAUpdate(BaseModel):
    """Structured representation of a single DGCA publication update."""

    title: str
    category: str
    published_date: str
    summary: str
    key_changes: List[str] = Field(default_factory=list)
    effective_date: str = "Not specified in the publication."
    impact_level: str = "MEDIUM"  # LOW | MEDIUM | HIGH | CRITICAL
    impacted_departments: List[str] = Field(default_factory=list)
    recommended_actions: List[str] = Field(default_factory=list)
    dgca_url: Optional[str] = None
    pdf_url: Optional[str] = None

    def slug(self) -> str:
        s = re.sub(r"[^a-z0-9]+", "-", self.title.lower()).strip("-")
        return s[:60] or "dgca-update"


IMPACT_COLORS = {
    "LOW": "#15803d",
    "MEDIUM": "#b45309",
    "HIGH": "#b91c1c",
    "CRITICAL": "#7f1d1d",
}

print("DGCAUpdate model ready.")


## 6. Test DGCA Update

The same hand-crafted example publication as before, with dummy links.


In [ ]:
test_update = DGCAUpdate(
    title="DGCA Example Regulation — Revision 2",
    category="Regulation",
    published_date="17 August 2026",
    summary=(
        "DGCA has issued Revision 2 of this regulation, updating flight duty "
        "time limitations and introducing revised documentation requirements "
        "for operators. This is example/test data used to validate the email "
        "notification pipeline."
    ),
    key_changes=[
        "Flight duty period limits revised for augmented crew operations.",
        "New mandatory documentation format for duty time records.",
        "Clarified reporting timelines for fatigue-related occurrences.",
    ],
    effective_date="1 October 2026",
    impact_level="HIGH",
    impacted_departments=["Flight Operations", "Safety", "Regulatory Affairs"],
    recommended_actions=[
        "Review current duty time records against the new documentation format.",
        "Brief Flight Operations and Safety teams on revised FDTL limits.",
        "Update internal SOPs ahead of the effective date.",
    ],
    dgca_url="https://www.dgca.gov.in/digigov-portal/ (example link)",
    pdf_url="https://example.com/dummy-dgca-regulation-revision-2.pdf",
)

print(test_update.model_dump_json(indent=2))


## 7. AI Summary Generation

Unchanged from the original design: answers what/what-changed/why/when/who/
action-to-take. Uses OpenAI if `OPENAI_API_KEY` is configured, otherwise a
deterministic template built from the structured fields — so this step never
silently pretends an unconfigured API "worked".


In [ ]:
def _fallback_summary(update: DGCAUpdate) -> str:
    """Deterministic, no-API summary built straight from structured fields."""
    departments = ", ".join(update.impacted_departments) or "Not specified in the publication."
    actions = "; ".join(update.recommended_actions) or "Not specified in the publication."
    return (
        f"{update.title} ({update.category}, published {update.published_date}). "
        f"{update.summary} "
        f"Effective date: {update.effective_date}. Impact level: {update.impact_level}. "
        f"Impacted departments: {departments}. "
        f"Recommended actions: {actions}"
    )


def generate_ai_summary(update: DGCAUpdate, config: PipelineConfig) -> str:
    if not (_OPENAI_SDK_AVAILABLE and config.openai_configured()):
        logger.warning("OPENAI_API_KEY not configured -- using fallback template summary.")
        return _fallback_summary(update)

    prompt = (
        "You are a regulatory intelligence analyst for an airline.\n"
        "Summarize the following DGCA publication for an executive audience. Answer\n"
        "concisely: what it is about, what changed, why it matters, when it becomes\n"
        "effective, who is impacted, and what action to consider. If information is\n"
        "not available, say \"Not specified in the publication.\" Do not invent facts.\n\n"
        f"Title: {update.title}\n"
        f"Category: {update.category}\n"
        f"Published: {update.published_date}\n"
        f"Effective date: {update.effective_date}\n"
        f"Impact level: {update.impact_level}\n"
        f"Raw summary: {update.summary}\n"
        f"Key changes: {'; '.join(update.key_changes)}\n"
        f"Impacted departments: {', '.join(update.impacted_departments)}\n"
        f"Recommended actions: {'; '.join(update.recommended_actions)}\n"
    )
    try:
        client = OpenAI(api_key=config.openai_api_key)
        response = client.chat.completions.create(
            model=config.llm_model,
            messages=[
                {"role": "system", "content": "You write concise, accurate executive summaries for aviation regulatory updates."},
                {"role": "user", "content": prompt},
            ],
            temperature=0.3,
        )
        text = (response.choices[0].message.content or "").strip()
        if not text:
            raise ValueError("Empty response from LLM")
        logger.info(f"AI summary generated via OpenAI model '{config.llm_model}'.")
        return text
    except Exception as exc:
        logger.error(f"OpenAI summary generation failed ({exc.__class__.__name__}). Falling back to template summary.")
        return _fallback_summary(update)


print("AI summary function ready.")


## 8. HTML Email Generation

Unchanged from the original design: a table-based, inline-CSS template
(Outlook's rendering engine is Word-based, so flexbox/grid and external
stylesheets are avoided). **This is the one function every delivery provider
shares** — `render_email_html()` doesn't know or care which `EmailSender`
will eventually receive its output, which is exactly what keeps generation
and delivery decoupled.


In [ ]:
EMAIL_TEMPLATE = Template(r'''
<!DOCTYPE html>
<html>
<head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1.0"></head>
<body style="margin:0;padding:0;background-color:#f2f4f7;font-family:Segoe UI, Arial, sans-serif;">
{% if test_mode %}
<table role="presentation" width="100%" cellpadding="0" cellspacing="0" style="background-color:#fff3cd;">
  <tr><td style="padding:8px 24px;font-size:12px;color:#664d03;text-align:center;">
    TEST MODE -- This is a sample notification generated for pipeline testing.
  </td></tr>
</table>
{% endif %}
<table role="presentation" width="100%" cellpadding="0" cellspacing="0">
  <tr>
    <td align="center" style="padding:24px 12px;">
      <table role="presentation" width="640" cellpadding="0" cellspacing="0" style="background-color:#ffffff;border:1px solid #d0d5dd;border-radius:6px;overflow:hidden;">

        <tr>
          <td style="background-color:#0b2a4a;padding:20px 28px;">
            <span style="color:#ffffff;font-size:16px;font-weight:600;letter-spacing:0.5px;">DGCA REGULATORY INTELLIGENCE UPDATE</span>
          </td>
        </tr>

        <tr>
          <td style="padding:24px 28px 8px 28px;">
            <span style="font-size:19px;font-weight:600;color:#101828;">{{ update.title }}</span>
          </td>
        </tr>

        <tr>
          <td style="padding:4px 28px 20px 28px;">
            <table role="presentation" cellpadding="0" cellspacing="0" style="font-size:13px;color:#475467;">
              <tr>
                <td style="padding:3px 16px 3px 0;font-weight:600;color:#344054;">Category</td>
                <td style="padding:3px 0;">{{ update.category }}</td>
              </tr>
              <tr>
                <td style="padding:3px 16px 3px 0;font-weight:600;color:#344054;">Published Date</td>
                <td style="padding:3px 0;">{{ update.published_date }}</td>
              </tr>
              <tr>
                <td style="padding:3px 16px 3px 0;font-weight:600;color:#344054;">Effective Date</td>
                <td style="padding:3px 0;">{{ update.effective_date }}</td>
              </tr>
              <tr>
                <td style="padding:3px 16px 3px 0;font-weight:600;color:#344054;">Impact Level</td>
                <td style="padding:3px 0;">
                  <span style="background-color:{{ impact_color }};color:#ffffff;padding:2px 10px;border-radius:3px;font-size:12px;font-weight:600;">{{ update.impact_level }}</span>
                </td>
              </tr>
            </table>
          </td>
        </tr>

        <tr><td style="border-top:1px solid #eaecf0;"></td></tr>

        <tr>
          <td style="padding:20px 28px 8px 28px;">
            <span style="font-size:14px;font-weight:700;color:#101828;">EXECUTIVE SUMMARY</span>
            <p style="font-size:14px;line-height:1.55;color:#344054;margin:10px 0 0 0;">{{ ai_summary }}</p>
          </td>
        </tr>

        {% if update.key_changes %}
        <tr>
          <td style="padding:16px 28px 8px 28px;">
            <span style="font-size:14px;font-weight:700;color:#101828;">KEY CHANGES</span>
            <ul style="margin:10px 0 0 0;padding-left:20px;font-size:14px;line-height:1.6;color:#344054;">
              {% for item in update.key_changes %}<li>{{ item }}</li>{% endfor %}
            </ul>
          </td>
        </tr>
        {% endif %}

        {% if update.impacted_departments %}
        <tr>
          <td style="padding:16px 28px 8px 28px;">
            <span style="font-size:14px;font-weight:700;color:#101828;">IMPACTED DEPARTMENTS</span>
            <ul style="margin:10px 0 0 0;padding-left:20px;font-size:14px;line-height:1.6;color:#344054;">
              {% for dept in update.impacted_departments %}<li>{{ dept }}</li>{% endfor %}
            </ul>
          </td>
        </tr>
        {% endif %}

        {% if update.recommended_actions %}
        <tr>
          <td style="padding:16px 28px 8px 28px;">
            <span style="font-size:14px;font-weight:700;color:#101828;">RECOMMENDED ACTIONS</span>
            <ul style="margin:10px 0 0 0;padding-left:20px;font-size:14px;line-height:1.6;color:#344054;">
              {% for action in update.recommended_actions %}<li>{{ action }}</li>{% endfor %}
            </ul>
          </td>
        </tr>
        {% endif %}

        <tr><td style="border-top:1px solid #eaecf0;padding-top:4px;"></td></tr>

        <tr>
          <td style="padding:20px 28px 28px 28px;">
            <span style="font-size:14px;font-weight:700;color:#101828;">DOCUMENT LINKS</span><br/>
            <table role="presentation" cellpadding="0" cellspacing="0" style="margin-top:12px;">
              <tr>
                <td style="padding-right:10px;padding-bottom:10px;">
                  <a href="{{ update.dgca_url }}" style="display:inline-block;background-color:#0b2a4a;color:#ffffff;text-decoration:none;font-size:13px;padding:9px 16px;border-radius:4px;">View DGCA Publication</a>
                </td>
                <td style="padding-right:10px;padding-bottom:10px;">
                  <a href="{{ update.pdf_url }}" style="display:inline-block;background-color:#0b2a4a;color:#ffffff;text-decoration:none;font-size:13px;padding:9px 16px;border-radius:4px;">Download PDF</a>
                </td>
                <td style="padding-bottom:10px;">
                  <a href="#" style="display:inline-block;background-color:#98a2b3;color:#ffffff;text-decoration:none;font-size:13px;padding:9px 16px;border-radius:4px;">Open Dashboard (Phase 8)</a>
                </td>
              </tr>
            </table>
          </td>
        </tr>

        <tr>
          <td style="background-color:#f9fafb;padding:14px 28px;border-top:1px solid #eaecf0;">
            <span style="font-size:11px;color:#98a2b3;">
              DGCA Regulatory Intelligence and Monitoring System -- automated notification generated {{ generated_at }}.
              This is {{ "test/preview" if test_mode else "production" }} output. Verify against the official DGCA publication before acting.
            </span>
          </td>
        </tr>

      </table>
    </td>
  </tr>
</table>
</body>
</html>
''')


def render_email_html(update: DGCAUpdate, ai_summary: str, test_mode: bool) -> str:
    return EMAIL_TEMPLATE.render(
        update=update,
        ai_summary=ai_summary,
        impact_color=IMPACT_COLORS.get(update.impact_level.upper(), "#475467"),
        test_mode=test_mode,
        generated_at=datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC"),
    )


print("Email template ready.")


## 9. Email Validation

Before any provider (even the local preview) gets the generated HTML, we
confirm every field the project brief requires is actually present in it.
This catches template regressions early -- e.g. a future template edit that
accidentally drops a section -- rather than "sending" a broken-looking email.


In [ ]:
@dataclass
class EmailValidationResult:
    is_valid: bool
    checked: List[str] = field(default_factory=list)
    missing: List[str] = field(default_factory=list)


def validate_email_html(html_body: str, update: DGCAUpdate, ai_summary: str) -> EmailValidationResult:
    checks = {
        "Publication title": update.title in html_body,
        "Category": update.category in html_body,
        "Published date": update.published_date in html_body,
        "Executive summary": bool(ai_summary.strip()) and ai_summary in html_body,
        "Impact level": update.impact_level in html_body,
        "DGCA publication link": bool(update.dgca_url) and update.dgca_url in html_body,
        "PDF link": bool(update.pdf_url) and update.pdf_url in html_body,
    }
    for i, change in enumerate(update.key_changes, start=1):
        checks[f"Key change #{i}"] = change in html_body
    for i, dept in enumerate(update.impacted_departments, start=1):
        checks[f"Impacted department #{i}"] = dept in html_body
    for i, action in enumerate(update.recommended_actions, start=1):
        checks[f"Recommended action #{i}"] = action in html_body

    checked = list(checks.keys())
    missing = [name for name, present in checks.items() if not present]
    result = EmailValidationResult(is_valid=(len(missing) == 0), checked=checked, missing=missing)

    if result.is_valid:
        logger.info(f"Email validation PASSED: all {len(checked)} required elements present.")
    else:
        logger.error(f"Email validation FAILED: missing {missing}")
    return result


print("Email validation function ready.")


## 10. Email Delivery Abstraction — the `EmailSender` Interface

This is the seam that decouples generation from delivery. Every provider
(local preview, Microsoft Graph, SMTP, and any future `api` provider)
implements the same two-method interface, so `send_dgca_notification()`
(Section 17) never needs to know which one it's talking to.


In [ ]:
@dataclass
class EmailSendResult:
    success: bool
    provider: str
    message: str
    detail: Optional[dict] = None


class EmailSender(ABC):
    """Common interface every email delivery provider must implement."""

    provider_name: str = "unknown"

    @abstractmethod
    def is_configured(self) -> bool:
        """Return True if this provider has everything it needs to actually deliver."""
        raise NotImplementedError

    @abstractmethod
    def send(self, subject: str, html_body: str, recipient: str) -> EmailSendResult:
        """Deliver (or simulate delivering) an email. Implementations must catch
        their own exceptions and return a failed EmailSendResult instead of
        raising, so the orchestrator never needs a provider-specific try/except."""
        raise NotImplementedError


T = TypeVar("T")


def retry_with_backoff(fn: Callable[[], T], max_attempts: int = 3, base_delay: float = 1.0) -> T:
    """Retry a zero-arg callable on exception, with exponential backoff.
    Shared by any provider that talks to a real network service."""
    last_exc: Optional[Exception] = None
    for attempt in range(1, max_attempts + 1):
        try:
            return fn()
        except Exception as exc:
            last_exc = exc
            if attempt == max_attempts:
                break
            delay = base_delay * (2 ** (attempt - 1))
            logger.warning(f"Attempt {attempt}/{max_attempts} failed ({exc.__class__.__name__}). Retrying in {delay:.1f}s...")
            time.sleep(delay)
    raise last_exc


print("EmailSender interface + retry_with_backoff ready.")


## 11. Provider — Local Preview (default, always available)

No network calls, no credentials, cannot fail for configuration reasons.
Writes the exact same HTML any other provider would have sent to a local
file that can be opened directly in a browser. This is `EMAIL_PROVIDER`'s
default value, and also the automatic fallback whenever `TEST_MODE=true`
prevents a real provider from actually delivering.


In [ ]:
class PreviewEmailSender(EmailSender):
    """Always-available provider: saves the rendered HTML locally instead of
    sending it anywhere."""

    provider_name = "preview"

    def __init__(self, preview_dir: Path):
        self.preview_dir = preview_dir

    def is_configured(self) -> bool:
        return True

    def send(self, subject: str, html_body: str, recipient: str) -> EmailSendResult:
        self.preview_dir.mkdir(parents=True, exist_ok=True)
        timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        slug = re.sub(r"[^a-z0-9]+", "-", subject.lower()).strip("-")[:60] or "email"
        path = self.preview_dir / f"{timestamp}_{slug}.html"
        path.write_text(html_body, encoding="utf-8")
        logger.info(f"[preview] Saved local HTML preview (no network call made): {path}")
        return EmailSendResult(
            success=True,
            provider=self.provider_name,
            message=f"Saved local preview. Would have gone to: {recipient}",
            detail={"path": str(path)},
        )


print("PreviewEmailSender ready.")


## 12. Provider — Microsoft Graph / Outlook (planned production provider)

**This is the planned production provider — implemented in full, but entirely
optional at this stage.** It is never instantiated unless `EMAIL_PROVIDER=
microsoft_graph`, and even then it only actually sends when `TEST_MODE=false`
*and* `config.graph_configured()` is true.

### How to plug this in later (no code changes needed)

1. Create an Azure AD App Registration with the **Application permission**
   `Mail.Send` for Microsoft Graph, admin-consented (see the project's Phase 1
   discussion for the full step-by-step — Azure Portal > Entra ID > App
   registrations).
2. Create a local `config/.env` and fill in `AZURE_CLIENT_ID`,
   `AZURE_TENANT_ID`, `AZURE_CLIENT_SECRET`, `OUTLOOK_SENDER_EMAIL`,
   `OUTLOOK_RECIPIENT_EMAIL`.
3. Set `EMAIL_PROVIDER=microsoft_graph` and `TEST_MODE=false`.
4. Re-run this notebook. Nothing else changes — the same `DGCAUpdate`, the
   same `render_email_html()` output, the same validation step all feed into
   this provider instead of the preview one.

### Auth approach (unchanged from the original design)

OAuth 2.0 **Client Credentials flow** (app-only auth) via `msal`, since this
pipeline runs unattended: a confidential client acquires a token for
`https://graph.microsoft.com/.default`, used to call
`POST /v1.0/users/{sender}/sendMail`. Delegated/device-code auth was rejected
because it needs an interactive sign-in on every run.


In [ ]:
class MicrosoftGraphEmailSender(EmailSender):
    """Production provider: Microsoft Graph /sendMail via OAuth2 client-credentials
    (app-only) auth. See the Section 12 markdown above for the plug-in steps."""

    provider_name = "microsoft_graph"
    GRAPH_BASE_URL = "https://graph.microsoft.com/v1.0"

    def __init__(self, config: PipelineConfig):
        self.config = config

    def is_configured(self) -> bool:
        return self.config.graph_configured()

    def _get_access_token(self) -> Optional[str]:
        try:
            app = msal.ConfidentialClientApplication(
                client_id=self.config.azure_client_id,
                authority=f"https://login.microsoftonline.com/{self.config.azure_tenant_id}",
                client_credential=self.config.azure_client_secret,
            )
            result = retry_with_backoff(
                lambda: app.acquire_token_for_client(scopes=["https://graph.microsoft.com/.default"])
            )
            if result and "access_token" in result:
                logger.info("[microsoft_graph] Authentication successful (token acquired -- value not displayed).")
                return result["access_token"]
            error = (result or {}).get("error", "unknown_error")
            description = (result or {}).get("error_description", "No description provided.")
            logger.error(f"[microsoft_graph] Authentication failed: {error} -- {description}")
            return None
        except Exception as exc:
            logger.error(f"[microsoft_graph] Exception during authentication: {exc.__class__.__name__}: {exc}")
            return None

    def send(self, subject: str, html_body: str, recipient: str) -> EmailSendResult:
        if not self.is_configured():
            return EmailSendResult(False, self.provider_name, "Azure/Outlook configuration is incomplete.")

        access_token = self._get_access_token()
        if not access_token:
            return EmailSendResult(False, self.provider_name, "Microsoft Graph authentication failed. See log for details.")

        url = f"{self.GRAPH_BASE_URL}/users/{self.config.outlook_sender_email}/sendMail"
        headers = {"Authorization": f"Bearer {access_token}", "Content-Type": "application/json"}
        payload = {
            "message": {
                "subject": subject,
                "body": {"contentType": "HTML", "content": html_body},
                "toRecipients": [{"emailAddress": {"address": recipient}}],
            },
            "saveToSentItems": "true",
        }
        try:
            response = retry_with_backoff(lambda: requests.post(url, headers=headers, json=payload, timeout=30))
        except Exception as exc:
            logger.error(f"[microsoft_graph] Exception while sending: {exc.__class__.__name__}: {exc}")
            return EmailSendResult(False, self.provider_name, f"Exception during send: {exc}")

        if response.status_code == 202:
            logger.info("[microsoft_graph] SUCCESS: email sent (HTTP 202 Accepted).")
            return EmailSendResult(True, self.provider_name, "Email sent successfully (HTTP 202 Accepted).")
        logger.error(f"[microsoft_graph] sendMail failed: HTTP {response.status_code} -- {response.text[:300]}")
        return EmailSendResult(False, self.provider_name, f"Graph API returned HTTP {response.status_code}.")


print("MicrosoftGraphEmailSender ready (implemented, optional -- inactive without Azure config).")


## 13. Provider — SMTP (development-friendly alternative)

A generic SMTP provider using only Python's standard library (`smtplib`) --
no new dependency. Useful for pointing at a local dev mail catcher (MailHog,
Mailpit, smtp4dev) to see real "delivery" happen without any cloud
credentials, or at a real SMTP endpoint if that's ever preferred over Graph.


In [ ]:
class SMTPEmailSender(EmailSender):
    """Generic SMTP provider -- an alternative to Microsoft Graph that needs
    no Azure setup, useful for local dev mail catchers."""

    provider_name = "smtp"

    def __init__(self, config: PipelineConfig):
        self.config = config

    def is_configured(self) -> bool:
        return self.config.smtp_configured()

    def send(self, subject: str, html_body: str, recipient: str) -> EmailSendResult:
        if not self.is_configured():
            return EmailSendResult(False, self.provider_name, "SMTP configuration is incomplete (SMTP_HOST / sender email).")

        message = MIMEMultipart("alternative")
        message["Subject"] = subject
        message["From"] = self.config.outlook_sender_email
        message["To"] = recipient
        message.attach(MIMEText(html_body, "html", "utf-8"))

        def _do_send() -> None:
            with smtplib.SMTP(self.config.smtp_host, self.config.smtp_port, timeout=20) as server:
                if self.config.smtp_use_tls:
                    server.starttls()
                if self.config.smtp_username:
                    server.login(self.config.smtp_username, self.config.smtp_password or "")
                server.sendmail(self.config.outlook_sender_email, [recipient], message.as_string())

        try:
            retry_with_backoff(_do_send, max_attempts=2)
        except Exception as exc:
            logger.error(f"[smtp] Send failed: {exc.__class__.__name__}: {exc}")
            return EmailSendResult(False, self.provider_name, f"SMTP send failed: {exc}")

        logger.info(f"[smtp] SUCCESS: email sent via {self.config.smtp_host}:{self.config.smtp_port}.")
        return EmailSendResult(True, self.provider_name, "Email sent successfully via SMTP.")


print("SMTPEmailSender ready (optional -- inactive without a reachable SMTP server).")


## 14. Provider Selection — `get_email_sender()`

The factory that turns `EMAIL_PROVIDER` into a concrete `EmailSender`. An
unknown provider name, or the reserved-but-unimplemented `"api"` value,
raises a clear exception rather than silently doing nothing -- handled by
the orchestrator in Section 17, not swallowed here.


In [ ]:
def get_email_sender(config: PipelineConfig) -> EmailSender:
    provider = config.email_provider.strip().lower()
    if provider == "preview":
        return PreviewEmailSender(PREVIEW_DIR)
    if provider == "microsoft_graph":
        return MicrosoftGraphEmailSender(config)
    if provider == "smtp":
        return SMTPEmailSender(config)
    if provider == "api":
        raise NotImplementedError(
            "EMAIL_PROVIDER=\'api\' is a reserved extension point for a future custom "
            "HTTP email API integration. Implement an EmailSender subclass and add it "
            "here when needed."
        )
    raise ValueError(f"Unknown EMAIL_PROVIDER \'{provider}\'. Expected one of: preview, microsoft_graph, smtp, api.")


print("Provider factory ready.")


## 15. Logging

Console output plus a per-day log file under `logs/`. Never logs secret
values -- only `SET`/`MISSING` booleans or non-sensitive status text,
enforced throughout every function above.


In [ ]:
LOG_FILE = LOG_DIR / f"email_pipeline_{datetime.now(timezone.utc).strftime('%Y%m%d')}.log"

logger = logging.getLogger("dgca_email_pipeline")
logger.setLevel(logging.INFO)
logger.handlers.clear()  # avoid duplicate handlers on notebook re-run

_formatter = logging.Formatter("[%(asctime)s] [%(levelname)s] %(message)s", datefmt="%Y-%m-%d %H:%M:%S")

_console_handler = logging.StreamHandler()
_console_handler.setFormatter(_formatter)
logger.addHandler(_console_handler)

_file_handler = logging.FileHandler(LOG_FILE, encoding="utf-8")
_file_handler.setFormatter(_formatter)
logger.addHandler(_file_handler)

logger.propagate = False

logger.info("Logger initialized.")
logger.info(f"Log file: {LOG_FILE}")


## 16. Error Handling

Three real, non-mocked failure scenarios:

1. An unconfigured `MicrosoftGraphEmailSender` must fail gracefully (return a
   failed `EmailSendResult`, never raise).
2. An unconfigured `SMTPEmailSender` must do the same.
3. An unknown `EMAIL_PROVIDER` value must raise a clear, catchable exception
   from `get_email_sender()` rather than silently returning something wrong.


In [ ]:
_graph_probe = MicrosoftGraphEmailSender(config)
_graph_result = _graph_probe.send("probe", "<html></html>", "someone@example.com")
assert _graph_result.success is False
print(f"MicrosoftGraphEmailSender graceful-failure check passed: {_graph_result.message}")

_smtp_probe = SMTPEmailSender(PipelineConfig(smtp_host=None))
_smtp_result = _smtp_probe.send("probe", "<html></html>", "someone@example.com")
assert _smtp_result.success is False
print(f"SMTPEmailSender graceful-failure check passed: {_smtp_result.message}")

try:
    get_email_sender(PipelineConfig(email_provider="not_a_real_provider"))
    raise AssertionError("Expected ValueError for an unknown provider.")
except ValueError as exc:
    print(f"Unknown-provider check passed: {exc}")


## 17. Orchestrator & End-to-End Test

`send_dgca_notification()` ties everything together: AI summary -> HTML
render -> validation -> provider dispatch. If `TEST_MODE=true`, any
non-preview provider is swapped for the preview provider before dispatch --
so the flow below runs safely all the way through with **zero external
calls**, using only the default configuration (no `config/.env` required).


In [ ]:
@dataclass
class PipelineResult:
    success: bool
    provider: str
    mode: str
    message: str
    preview_path: Optional[str] = None
    validation_passed: bool = False
    run_id: str = field(default_factory=lambda: str(uuid.uuid4())[:8])
    timestamp: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())


def send_dgca_notification(update: DGCAUpdate, config: PipelineConfig) -> PipelineResult:
    logger.info(f"Starting DGCA notification pipeline for: '{update.title}'")

    ai_summary = generate_ai_summary(update, config)
    subject = f"DGCA Regulatory Update: {update.title} [{update.impact_level}]"

    requested_provider = config.email_provider.strip().lower()
    is_preview_render = config.test_mode or requested_provider == "preview"
    html_body = render_email_html(update, ai_summary, test_mode=is_preview_render)

    validation = validate_email_html(html_body, update, ai_summary)
    if not validation.is_valid:
        return PipelineResult(
            success=False,
            provider=requested_provider,
            mode="validation-failed",
            message=f"Email failed validation, not sending. Missing: {validation.missing}",
            validation_passed=False,
        )

    try:
        sender = get_email_sender(config)
    except (ValueError, NotImplementedError) as exc:
        logger.error(f"Could not create email sender for provider '{requested_provider}': {exc}")
        return PipelineResult(
            success=False, provider=requested_provider, mode="config-error",
            message=str(exc), validation_passed=True,
        )

    if sender.provider_name != "preview" and config.test_mode:
        logger.info(
            f"[TEST_MODE] EMAIL_PROVIDER='{requested_provider}' selected, but TEST_MODE=true -- "
            "falling back to local preview instead of a real send."
        )
        sender = PreviewEmailSender(PREVIEW_DIR)

    recipient = config.outlook_recipient_email or "unset-recipient@example.com"
    send_result = sender.send(subject=subject, html_body=html_body, recipient=recipient)

    return PipelineResult(
        success=send_result.success,
        provider=send_result.provider,
        mode="preview" if send_result.provider == "preview" else "live",
        message=send_result.message,
        preview_path=(send_result.detail or {}).get("path") if send_result.detail else None,
        validation_passed=True,
    )


result = send_dgca_notification(test_update, config)
result


## 18. Results

In [ ]:
print("=" * 60)
print("PIPELINE RUN RESULT")
print("=" * 60)
print(f"Run ID              : {result.run_id}")
print(f"Timestamp           : {result.timestamp}")
print(f"Requested provider  : {config.email_provider}")
print(f"Actual provider used: {result.provider}")
print(f"Mode                : {result.mode}")
print(f"Validation passed   : {result.validation_passed}")
print(f"Success             : {result.success}")
print(f"Message             : {result.message}")
if result.preview_path:
    print(f"Preview file        : {result.preview_path}")
print("=" * 60)

if result.preview_path:
    from IPython.display import HTML, display
    display(HTML(Path(result.preview_path).read_text(encoding="utf-8")))


## 19. Next Steps

**Phase 1 status: email generation and delivery are now fully decoupled, and
the pipeline runs end-to-end with zero external dependencies** (default
`EMAIL_PROVIDER=preview`, `TEST_MODE=true`, no `config/.env` needed at all).

### To switch delivery providers later

- **Local dev SMTP catcher** (see real "delivery" without any cloud setup):
  run a tool like MailHog/Mailpit locally, set `SMTP_HOST`/`SMTP_PORT` in
  `config/.env`, set `EMAIL_PROVIDER=smtp` and `TEST_MODE=false`.
- **Microsoft Graph / Outlook (production)**: follow the steps in Section 12
  once an Azure AD App Registration with `Mail.Send` is approved. Nothing in
  this notebook needs to change beyond `config/.env` values.
- **A future custom API provider**: implement an `EmailSender` subclass (same
  pattern as `SMTPEmailSender`) and add it to `get_email_sender()`; the rest
  of the pipeline is unaffected.

### Not started yet (by design)

- Phase 3: PostgreSQL change-detection storage
- Phase 4: PDF download & text extraction
- Phase 5: Old-vs-new AI comparison
- Phase 6: Structured AI summary integration into the scraper flow
- Phase 7: Agentic workflow orchestration (LangGraph, if it earns its place)
- Phase 8: Streamlit dashboard

Phase 2 (the DGCA "What's New" scraper) is already built and tested
separately in `02_DGCA_Website_Monitor.ipynb`. Waiting for confirmation
before starting Phase 3.
